# CareMate — Integrated App (UI + Real Backend)

This notebook connects your **Gradio UI** (`ui_with_pdf___2_.ipynb`) to your
**real trained pipeline** (`07_Deployement.ipynb`): Whisper ASR + your
disease classifier. Run the cells top to bottom in Google Colab.

**What changed vs. your two source notebooks:**
- The UI (`app.py`, CSS) is unchanged except one import line.
- `backend/dummy_pipeline.py` is replaced by `backend/pipeline.py`, which
  loads your real Whisper model + `Best_Disease_Model` classifier and
  returns real predictions instead of placeholder text.
- Everything is written into your existing `CareMate_AI` project folder on
  Drive, next to your `models/` directory, so the classifier is found
  automatically.

**Requirement:** your Drive must already contain
`CareMate_AI/models/Best_Disease_Model/` (with the tokenizer, model weights,
and `label_encoder.pkl`) — the same folder your deployment notebook loads
from.

## 1. Mount Google Drive

In [1]:
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


## 2. Install dependencies

In [2]:
!pip install -q gradio==5.38.2
!pip install -q openai-whisper
!pip install -q transformers
!pip install -q sentencepiece
!pip install -q accelerate
!pip install -q joblib
!pip install -q pandas

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.6/68.6 kB 2.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.5/59.5 MB 15.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 324.5/324.5 kB 21.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 444.8/444.8 kB 25.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 45.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.4/11.4 MB 97.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 74.3/74.3 kB 4.9 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
python-fasthtml 0.14.6 requires starlette>=1.0.1, but you have starlette 0.52.1 which is incompatible.
google-genai 2.11.0 requires pydantic<3.0.0,>=2.12.5, but you have pydantic 2.11.10 which is incompatible.
google-adk 2.4.0 requires pydantic<3,>=2.12, but you have py

## 3. Project paths

`PROJECT_PATH` must point to the same `CareMate_AI` folder your deployment
notebook used — that's where `models/Best_Disease_Model` already lives.
The UI is written into `PROJECT_PATH/frontend`, and `backend/pipeline.py`
looks one level up (`../models`) to find your trained model.

In [3]:
import os

PROJECT_PATH = "/content/drive/MyDrive/CareMate_AI"
FRONTEND_DIR = os.path.join(PROJECT_PATH, "frontend")

for folder in ["css", "backend", "assets", "samples", "outputs", "components"]:
    os.makedirs(os.path.join(FRONTEND_DIR, folder), exist_ok=True)

os.chdir(FRONTEND_DIR)
print("Working directory:", os.getcwd())

Working directory: /content/drive/MyDrive/CareMate_AI/frontend


## 4. Sanity-check that your trained model is where the backend expects it

If this cell errors or prints an empty list, fix `PROJECT_PATH` above (or
move your `models/` folder) before continuing — the backend will fail to
load otherwise.

In [4]:
MODEL_CHECK_PATH = os.path.join(PROJECT_PATH, "models", "Best_Disease_Model")

print("Looking for model at:", MODEL_CHECK_PATH)
print()
if os.path.isdir(MODEL_CHECK_PATH):
    print("Found. Contents:")
    print(os.listdir(MODEL_CHECK_PATH))
else:
    print("NOT FOUND. Fix PROJECT_PATH in the previous cell before continuing.")

Looking for model at: /content/drive/MyDrive/CareMate_AI/models/Best_Disease_Model

NOT FOUND. Fix PROJECT_PATH in the previous cell before continuing.


## 5. Write `backend/__init__.py`

In [5]:
%%writefile backend/__init__.py


Writing backend/__init__.py


## 6. Write `backend/pipeline.py` — the real backend

This is the piece that actually connects your two notebooks: it loads
Whisper and your disease classifier exactly like `07_Deployement.ipynb`
did, and exposes a `process(audio_path, output_language)` function with
the same return signature your UI already expects from
`backend/dummy_pipeline.py`. Read the module docstring below — it's
upfront about which parts (standardization, Hindi translation) are
best-effort placeholders vs. real model output.

In [6]:
%%writefile backend/pipeline.py
"""
CareMate - Real Backend Pipeline
=================================

This module replaces backend/dummy_pipeline.py. It wires the UI to the
actual models built in the deployment notebook:

    audio  --Whisper-->  original-language transcript + detected language
           --Whisper(translate)-->  English transcript
           --disease classifier-->  top-K disease predictions
           --rule-based advisor-->  recommendation text

CONTRACT (unchanged from dummy_pipeline.py — this is why app.py does not
need to change, only its import line does):

    from backend.pipeline import process

    process(audio_path: str, output_language: str) -> tuple[str, str, str, str, str, str]

    Return order:
        (
            detected_language,
            transcript,                 # original-language transcript
            standardized_transcript,    # English text actually fed to the model
            translated_transcript,      # transcript in the user's chosen output language
            clinical_summary,
            clinical_report_html,
        )

HONEST LIMITATIONS (please read before demoing this)
------------------------------------------------------
The deployment notebook this was built from only trains/loads two real
components: Whisper ASR and the disease-prediction classifier. There is
no dedicated clinical-NLP "standardization" model and no English->Hindi
translation model in this build yet. To stay honest instead of faking it:

  * `standardized_transcript` is the English text that is actually sent
    into the disease classifier (i.e. Whisper's English translation).
    It is not a separately-trained clinical normalizer.
  * `translated_transcript` is only genuinely translated when the
    requested output language is English (Whisper already gives us
    that for free) or when the audio's source language already matches
    the requested output language. If the person asks for Hindi output
    from, say, English or Bhojpuri audio, we fall back to showing the
    English transcript with a clear "[Hindi translation not available
    in this build]" note rather than inventing a translation.

Swap in real IndicTrans2 / clinical-standardization models later by
editing `_translate_to_output_language()` and `_standardize()` below —
nothing else in the UI needs to change.
"""

import os
import json
from datetime import datetime

import torch
import torch.nn.functional as F
import joblib
import whisper
from transformers import AutoTokenizer, AutoModelForSequenceClassification


# ---------------------------------------------------------------------------
# Paths
# ---------------------------------------------------------------------------
# Override with an environment variable if your Drive layout differs.
# Default matches the deployment notebook: models live directly under
# CareMate_AI/models/Best_Disease_Model, one level above the frontend/ folder.
PROJECT_PATH = os.environ.get(
    "CAREMATE_PROJECT_PATH",
    "/content/drive/MyDrive/CareMate_AI",
)

MODEL_PATH = os.path.join(PROJECT_PATH, "models", "Best_Disease_Model")
OUTPUT_PATH = os.path.join(PROJECT_PATH, "results")
HISTORY_FILE = os.path.join(OUTPUT_PATH, "patient_history.csv")

os.makedirs(OUTPUT_PATH, exist_ok=True)


# ---------------------------------------------------------------------------
# Model loading (happens once, at import time — matches deployment notebook)
# ---------------------------------------------------------------------------
print("=" * 50)
print("CareMate backend: loading models...")
print("=" * 50)

print("Loading Whisper (base)...")
_whisper_model = whisper.load_model("base")
print("Whisper loaded.")

print(f"Loading disease classifier from: {MODEL_PATH}")
_tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH, local_files_only=True)
_disease_model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_PATH, local_files_only=True
)
_disease_model.eval()

_encoder = joblib.load(os.path.join(MODEL_PATH, "label_encoder.pkl"))
print(f"Disease classifier loaded. {len(_encoder.classes_)} disease classes.")
print("=" * 50)
print("CareMate backend ready.")
print("=" * 50)


# ---------------------------------------------------------------------------
# Core model calls (same logic as the deployment notebook)
# ---------------------------------------------------------------------------
def predict_disease(text: str, top_k: int = 3):
    """Predict top-K diseases from English symptom text."""
    inputs = _tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        padding=True,
        max_length=128,
    )

    with torch.no_grad():
        outputs = _disease_model(**inputs)

    probabilities = F.softmax(outputs.logits, dim=1)
    top_probs, top_indices = torch.topk(probabilities, min(top_k, probabilities.shape[1]))

    predictions = []
    for prob, idx in zip(top_probs[0], top_indices[0]):
        disease = _encoder.inverse_transform([idx.item()])[0]
        predictions.append({
            "Disease": disease,
            "Confidence": round(prob.item() * 100, 2),
        })
    return predictions


def generate_recommendation(predictions):
    top = predictions[0]
    confidence = top["Confidence"]
    disease = top["Disease"]

    if confidence >= 80:
        advice = "High confidence prediction. Please consult a physician for proper diagnosis and treatment."
    elif confidence >= 50:
        advice = "Possible disease detected. Medical consultation is recommended."
    else:
        advice = "Low confidence prediction. More symptoms or clinical examination may be required."

    return {
        "Predicted Disease": disease,
        "Confidence": confidence,
        "Recommendation": advice,
    }


def format_predictions(predictions):
    output = ""
    for i, item in enumerate(predictions, start=1):
        output += f"{i}. {item['Disease']} ({item['Confidence']}%)\n"
    return output.strip()


def _language_name(code: str) -> str:
    try:
        return whisper.tokenizer.LANGUAGES.get(code, code).title()
    except Exception:
        return code


def _transcribe(audio_path: str):
    """
    Runs Whisper twice: once to get the original-language transcript
    (for the "Original Transcript" panel) and once with task="translate"
    to get the English text the disease model actually consumes.
    """
    original_result = _whisper_model.transcribe(audio_path, task="transcribe")
    english_result = _whisper_model.transcribe(audio_path, task="translate")

    detected_code = original_result.get("language", "en")
    detected_language = _language_name(detected_code)

    original_transcript = original_result["text"].strip()
    english_transcript = english_result["text"].strip()

    return detected_code, detected_language, original_transcript, english_transcript


def _translate_to_output_language(
    output_language: str,
    detected_code: str,
    original_transcript: str,
    english_transcript: str,
) -> str:
    """
    Best-effort "translated transcript" for the requested output language.
    See module docstring: there is no general-purpose translation model
    wired in yet, so this only returns a genuine translation for the
    cases we can actually support, and is explicit about the fallback.
    """
    wants_hindi = output_language.upper() == "HINDI"

    if not wants_hindi:
        # English requested — Whisper's translate task already gives us this.
        return english_transcript

    if detected_code == "hi":
        # Audio was already Hindi — the original transcript IS the Hindi version.
        return original_transcript

    # No EN->HI (or other language ->HI) model in this build yet.
    return (
        english_transcript
        + "\n\n[Hindi translation not available in this build — showing English text.]"
    )


def _clinical_report_html(disease: str, confidence: float, advice: str, summary_line: str) -> str:
    if confidence >= 80:
        priority = "High priority — please seek doctor review promptly."
        chip_class = "cm-chip-amber"
    elif confidence >= 50:
        priority = "Moderate priority — doctor review recommended."
        chip_class = "cm-chip-amber"
    else:
        priority = "Low confidence — more symptoms or examination may be needed."
        chip_class = "cm-chip-blue"

    return f"""
    <section class="cm-report">
      <div class="cm-report-header">
        <div>
          <span class="cm-report-kicker">Patient Report</span>
          <h3>{disease}</h3>
        </div>
        <div class="cm-score">
          <span>Confidence</span>
          <strong>{confidence}%</strong>
        </div>
      </div>

      <div class="cm-confidence" aria-label="Disease confidence score {confidence}%">
        <span style="width: {min(confidence, 100)}%"></span>
      </div>

      <div class="cm-report-grid">
        <article>
          <span class="cm-chip cm-chip-blue">Symptoms (from transcript)</span>
          <p>{summary_line}</p>
        </article>
        <article>
          <span class="cm-chip {chip_class}">Priority</span>
          <p>{priority}</p>
        </article>
      </div>

      <div class="cm-report-columns">
        <div>
          <h4>Precautions</h4>
          <ul>
            <li>Monitor symptoms and note any changes in severity.</li>
            <li>Stay hydrated and rested while awaiting doctor review.</li>
            <li>Seek urgent care if symptoms worsen suddenly or severely.</li>
          </ul>
        </div>
        <div>
          <h4>Recommendation</h4>
          <ul>
            <li>{advice}</li>
            <li>Do not self-medicate based only on this AI output.</li>
          </ul>
        </div>
      </div>

      <div class="cm-disclaimer">
        This AI report is not a diagnosis. Please visit a qualified doctor for confirmation and treatment.
      </div>
    </section>
    """


# ---------------------------------------------------------------------------
# Logging (same behavior as the deployment notebook's save_report / save_history)
# ---------------------------------------------------------------------------
def save_report(transcript, predictions, recommendation):
    report = {
        "Date": datetime.now().strftime("%d-%m-%Y"),
        "Time": datetime.now().strftime("%H:%M:%S"),
        "Transcript": transcript,
        "Predictions": predictions,
        "Recommendation": recommendation,
    }
    filename = "Patient_" + datetime.now().strftime("%Y%m%d_%H%M%S") + ".json"
    filepath = os.path.join(OUTPUT_PATH, filename)
    with open(filepath, "w", encoding="utf-8") as f:
        json.dump(report, f, indent=4, ensure_ascii=False)
    return filepath


def save_history(transcript, predictions):
    import pandas as pd

    top = predictions[0]
    row = {
        "Date": datetime.now().strftime("%d-%m-%Y"),
        "Time": datetime.now().strftime("%H:%M:%S"),
        "Transcript": transcript,
        "Disease": top["Disease"],
        "Confidence": top["Confidence"],
    }
    if os.path.exists(HISTORY_FILE):
        df = pd.read_csv(HISTORY_FILE)
    else:
        df = pd.DataFrame()
    df = pd.concat([df, pd.DataFrame([row])], ignore_index=True)
    df.to_csv(HISTORY_FILE, index=False)


# ---------------------------------------------------------------------------
# Public entry point — this is the only function app.py calls
# ---------------------------------------------------------------------------
def process(audio_path: str, output_language: str):
    if not audio_path or not os.path.exists(audio_path):
        empty_report = """
        <section class="cm-report cm-report-empty">
          <h3>No report yet</h3>
          <p>Upload or record a consultation, then process it to generate precautions, recommendations, confidence score, and doctor-review guidance.</p>
        </section>
        """
        return (
            "No audio detected",
            "No audio was provided. Please upload or record a consultation.",
            "-",
            "-",
            "-",
            empty_report,
        )

    detected_code, detected_language, original_transcript, english_transcript = _transcribe(audio_path)

    # This is the text actually fed into the disease classifier — labeled
    # here as "standardized" per the UI contract, see module docstring.
    standardized_transcript = english_transcript

    translated_transcript = _translate_to_output_language(
        output_language, detected_code, original_transcript, english_transcript
    )

    predictions = predict_disease(standardized_transcript, top_k=3)
    recommendation = generate_recommendation(predictions)
    prediction_text = format_predictions(predictions)

    clinical_summary = (
        f"Top predictions:\n{prediction_text}\n\n"
        f"Recommendation: {recommendation['Recommendation']}"
    )

    clinical_report_html = _clinical_report_html(
        disease=recommendation["Predicted Disease"],
        confidence=recommendation["Confidence"],
        advice=recommendation["Recommendation"],
        summary_line=standardized_transcript,
    )

    # Best-effort logging — never let a logging failure break the UI.
    try:
        save_report(original_transcript, predictions, recommendation)
        save_history(original_transcript, predictions)
    except Exception as e:
        print(f"[CareMate] Warning: could not save report/history: {e}")

    return (
        detected_language,
        original_transcript,
        standardized_transcript,
        translated_transcript,
        clinical_summary,
        clinical_report_html,
    )


Writing backend/pipeline.py


## 7. Write the CSS (unchanged from your UI notebook)

In [7]:
%%writefile css/style.css
/* ============================================================
   CAREMATE - PREMIUM CLINICAL DASHBOARD THEME
   Natural page scroll + dynamic, responsive card grid
   ============================================================ */

:root {
  --cm-bg-1: #020617;
  --cm-bg-2: #071330;
  --cm-bg-3: #000000;

  --cm-accent-1: #2563eb;
  --cm-accent-2: #38bdf8;
  --cm-accent-3: #14b8a6;
  --cm-warn: #f59e0b;
  --cm-danger: #fb7185;

  --cm-text-1: #f8faff;
  --cm-text-2: #dce9ff;
  --cm-text-3: #9cc9ff;

  --cm-card: rgba(15, 23, 42, 0.78);
  --cm-card-border: rgba(56, 189, 248, 0.18);
  --cm-card-border-strong: rgba(56, 189, 248, 0.4);
}

* {
  box-sizing: border-box;
}

body {
  margin: 0;
}

.gradio-container {
  background: radial-gradient(
      circle at 15% 0%,
      var(--cm-bg-2) 0%,
      var(--cm-bg-1) 45%,
      var(--cm-bg-3) 100%
    ) !important;
  font-family: "Inter", "Segoe UI", -apple-system, sans-serif !important;
  color: var(--cm-text-2) !important;
}

#cm-root {
  display: flex !important;
  flex-direction: column !important;
  padding: 16px 20px 28px 20px !important;
  gap: 14px !important;
  max-width: 1400px;
  margin: 0 auto;
}

#cm-header {
  text-align: center;
  padding: 8px 0 2px 0;
}

#cm-title {
  font-size: clamp(24px, 4vw, 40px);
  font-weight: 800;
  letter-spacing: 5px;
  color: var(--cm-text-1);
  text-shadow: 0 0 18px rgba(56, 189, 248, 0.35);
  margin: 0;
  line-height: 1.1;
}

#cm-subtitle {
  margin-top: 4px;
  font-size: clamp(9px, 1.4vw, 12px);
  font-weight: 500;
  letter-spacing: 3px;
  text-transform: uppercase;
  color: var(--cm-text-3);
}

.cm-section-label {
  text-transform: uppercase;
  letter-spacing: 1.6px;
  font-size: 11px;
  font-weight: 700;
  color: var(--cm-text-3);
  margin-bottom: 6px;
  display: block;
}

#cm-input-row > .row {
  flex-wrap: wrap !important;
  gap: 12px !important;
}

.cm-card {
  background: var(--cm-card) !important;
  border: 1px solid var(--cm-card-border) !important;
  border-radius: 14px !important;
  padding: 12px 14px !important;
  backdrop-filter: blur(18px);
  -webkit-backdrop-filter: blur(18px);
  box-shadow: 0 6px 24px rgba(0, 0, 0, 0.4);
  transition: border-color 0.2s ease, box-shadow 0.2s ease;
}

.cm-card:hover {
  border-color: var(--cm-card-border-strong) !important;
  box-shadow: 0 6px 30px rgba(37, 99, 235, 0.22);
}

#cm-audio .wrap,
#cm-audio [data-testid="audio"] {
  background: rgba(2, 6, 23, 0.5) !important;
  border: 1.5px dashed rgba(56, 189, 248, 0.3) !important;
  border-radius: 12px !important;
  min-height: 90px !important;
}

#cm-lang-selector .wrap {
  display: flex !important;
  gap: 6px;
  background: rgba(2, 6, 23, 0.55);
  padding: 4px;
  border-radius: 999px;
  border: 1px solid rgba(56, 189, 248, 0.18);
}

#cm-lang-selector label {
  border-radius: 999px !important;
  padding: 7px 18px !important;
  font-weight: 600 !important;
  font-size: 12px !important;
  letter-spacing: 0.5px;
  color: var(--cm-text-3) !important;
  transition: all 0.2s ease;
}

#cm-lang-selector input:checked + label,
#cm-lang-selector label.selected {
  background: linear-gradient(
    135deg,
    var(--cm-accent-1),
    var(--cm-accent-2)
  ) !important;
  color: #020617 !important;
  box-shadow: 0 0 14px rgba(56, 189, 248, 0.5);
}

#cm-detected-lang textarea {
  text-align: center !important;
  font-weight: 700 !important;
  font-size: 14px !important;
  color: var(--cm-accent-2) !important;
  letter-spacing: 1px;
  min-height: 44px !important;
  resize: none !important;
}

#cm-process-btn {
  display: block;
  margin: 4px auto !important;
  padding: 10px 44px !important;
  font-size: 13px !important;
  font-weight: 700 !important;
  letter-spacing: 2.5px;
  text-transform: uppercase;
  color: var(--cm-text-1) !important;
  background: linear-gradient(
    135deg,
    var(--cm-accent-1),
    var(--cm-accent-2)
  ) !important;
  border: none !important;
  border-radius: 999px !important;
  box-shadow: 0 0 20px rgba(37, 99, 235, 0.4);
  transition: transform 0.2s ease, box-shadow 0.2s ease;
}

#cm-process-btn:hover {
  transform: translateY(-1px);
  box-shadow: 0 0 28px rgba(56, 189, 248, 0.6);
}

#cm-outputs-row > .row {
  display: grid !important;
  grid-template-columns: repeat(auto-fit, minmax(260px, 1fr)) !important;
  gap: 14px !important;
}

.cm-output textarea {
  min-height: 160px !important;
  max-height: 420px !important;
  font-size: 13px !important;
  line-height: 1.5 !important;
  color: var(--cm-text-2) !important;
  overflow-y: auto !important;
  resize: vertical !important;
  padding: 8px 10px !important;
}

.gradio-container input,
.gradio-container textarea {
  background: rgba(2, 6, 23, 0.6) !important;
  border: 1px solid rgba(56, 189, 248, 0.15) !important;
  color: var(--cm-text-1) !important;
  border-radius: 10px !important;
}

#cm-report-card {
  padding: 0 !important;
  overflow: hidden;
}

.cm-report {
  background:
    linear-gradient(135deg, rgba(37, 99, 235, 0.18), rgba(20, 184, 166, 0.08)),
    rgba(2, 6, 23, 0.42);
  border: 1px solid rgba(56, 189, 248, 0.2);
  border-radius: 14px;
  padding: 18px;
  color: var(--cm-text-2);
}

.cm-report-header {
  display: flex;
  align-items: flex-start;
  justify-content: space-between;
  gap: 16px;
  margin-bottom: 12px;
}

.cm-report-kicker {
  color: var(--cm-text-3);
  display: block;
  font-size: 11px;
  font-weight: 800;
  letter-spacing: 1.8px;
  text-transform: uppercase;
  margin-bottom: 4px;
}

.cm-report h3,
.cm-report h4,
.cm-report p {
  margin: 0;
}

.cm-report h3 {
  color: var(--cm-text-1);
  font-size: 22px;
  line-height: 1.2;
}

.cm-score {
  min-width: 112px;
  text-align: right;
  background: rgba(2, 6, 23, 0.56);
  border: 1px solid rgba(56, 189, 248, 0.2);
  border-radius: 12px;
  padding: 10px 12px;
}

.cm-score span {
  color: var(--cm-text-3);
  display: block;
  font-size: 10px;
  font-weight: 800;
  letter-spacing: 1.2px;
  text-transform: uppercase;
}

.cm-score strong {
  color: #7dd3fc;
  display: block;
  font-size: 26px;
  line-height: 1.1;
}

.cm-confidence {
  height: 10px;
  background: rgba(15, 23, 42, 0.9);
  border-radius: 999px;
  overflow: hidden;
  margin-bottom: 14px;
  border: 1px solid rgba(148, 163, 184, 0.16);
}

.cm-confidence span {
  display: block;
  height: 100%;
  border-radius: inherit;
  background: linear-gradient(90deg, var(--cm-accent-3), var(--cm-accent-2));
  box-shadow: 0 0 18px rgba(56, 189, 248, 0.45);
}

.cm-report-grid,
.cm-report-columns {
  display: grid;
  gap: 12px;
}

.cm-report-grid {
  grid-template-columns: repeat(2, minmax(0, 1fr));
  margin-bottom: 14px;
}

.cm-report-grid article,
.cm-report-columns > div {
  background: rgba(15, 23, 42, 0.72);
  border: 1px solid rgba(148, 163, 184, 0.16);
  border-radius: 12px;
  padding: 12px;
}

.cm-chip {
  display: inline-flex;
  align-items: center;
  border-radius: 999px;
  font-size: 10px;
  font-weight: 800;
  letter-spacing: 1px;
  margin-bottom: 8px;
  padding: 5px 9px;
  text-transform: uppercase;
}

.cm-chip-blue {
  background: rgba(56, 189, 248, 0.14);
  color: #7dd3fc;
}

.cm-chip-amber {
  background: rgba(245, 158, 11, 0.14);
  color: #fcd34d;
}

.cm-report-columns {
  grid-template-columns: repeat(2, minmax(0, 1fr));
}

.cm-report h4 {
  color: var(--cm-text-1);
  font-size: 14px;
  margin-bottom: 8px;
}

.cm-report ul {
  margin: 0;
  padding-left: 18px;
}

.cm-report li {
  line-height: 1.45;
  margin-bottom: 6px;
}

.cm-disclaimer {
  margin-top: 14px;
  border: 1px solid rgba(251, 113, 133, 0.3);
  background: rgba(251, 113, 133, 0.1);
  color: #fecdd3;
  border-radius: 12px;
  padding: 12px;
  font-size: 13px;
  font-weight: 700;
}

.cm-report-empty {
  text-align: center;
  padding: 24px;
}

.cm-report-empty p {
  margin-top: 8px;
  color: var(--cm-text-3);
}

#cm-footer {
  text-align: center;
  padding: 10px 10px 4px 10px;
  font-size: 10px;
  letter-spacing: 1.5px;
  text-transform: uppercase;
  color: rgba(156, 201, 255, 0.4);
}

::-webkit-scrollbar {
  width: 7px;
}
::-webkit-scrollbar-thumb {
  background: rgba(56, 189, 248, 0.35);
  border-radius: 8px;
}

@media (max-width: 640px) {
  #cm-root {
    padding: 10px 12px 20px 12px;
  }
  #cm-outputs-row > .row,
  .cm-report-grid,
  .cm-report-columns {
    grid-template-columns: 1fr !important;
  }
  .cm-report-header {
    flex-direction: column;
  }
  .cm-score {
    text-align: left;
    width: 100%;
  }
}
/* CareMate report visibility fix only */
#cm-report-card .cm-report,
#cm-report-card .cm-report *,
.cm-report,
.cm-report * {
  opacity: 1 !important;
}

#cm-report-card .cm-report p,
#cm-report-card .cm-report li,
#cm-report-card .cm-report ul,
#cm-report-card .cm-report article p,
#cm-report-card .cm-report-columns li,
#cm-report-card .cm-report-grid p,
.cm-report p,
.cm-report li,
.cm-report ul,
.cm-report article p,
.cm-report-columns li,
.cm-report-grid p {
  color: #dce9ff !important;
  -webkit-text-fill-color: #dce9ff !important;
  font-size: 15px !important;
  font-weight: 600 !important;
  line-height: 1.55 !important;
}

#cm-report-card .cm-report li::marker,
.cm-report li::marker {
  color: #38bdf8 !important;
  -webkit-text-fill-color: #38bdf8 !important;
}

#cm-report-card .cm-report h3,
#cm-report-card .cm-report h4,
.cm-report h3,
.cm-report h4 {
  color: #f8faff !important;
  -webkit-text-fill-color: #f8faff !important;
}

#cm-report-card .cm-report-kicker,
.cm-report-kicker {
  color: #9cc9ff !important;
  -webkit-text-fill-color: #9cc9ff !important;
}

#cm-report-card .cm-disclaimer,
.cm-disclaimer {
  color: #ffe4e6 !important;
  -webkit-text-fill-color: #ffe4e6 !important;
}


/* ---- Process row: two buttons side by side ---- */
#cm-process-row > .row {
  display: flex !important;
  justify-content: center !important;
  align-items: center !important;
  gap: 16px !important;
  flex-wrap: wrap !important;
}

#cm-reset-btn {
  display: inline-block;
  padding: 10px 34px !important;
  font-size: 13px !important;
  font-weight: 700 !important;
  letter-spacing: 2px;
  text-transform: uppercase;
  color: var(--cm-text-2) !important;
  background: rgba(15, 23, 42, 0.4) !important;
  border: 1.5px solid rgba(56, 189, 248, 0.4) !important;
  border-radius: 999px !important;
  transition: all 0.2s ease;
}

#cm-reset-btn:hover {
  border-color: var(--cm-accent-2) !important;
  color: var(--cm-text-1) !important;
  box-shadow: 0 0 14px rgba(56, 189, 248, 0.25);
}

/* ---- Download button: one click, no file-card detour ---- */
#cm-download-btn {
  display: block;
  margin: 10px auto 0 auto !important;
  padding: 10px 30px !important;
  font-size: 13px !important;
  font-weight: 700 !important;
  letter-spacing: 2px;
  text-transform: uppercase;
  border-radius: 999px !important;
  background: linear-gradient(135deg, var(--cm-accent-3), var(--cm-accent-2)) !important;
  color: #04121f !important;
  border: none !important;
  box-shadow: 0 0 16px rgba(20, 184, 166, 0.35);
  transition: transform 0.2s ease, box-shadow 0.2s ease;
}

#cm-download-btn:hover {
  transform: translateY(-1px);
  box-shadow: 0 0 24px rgba(20, 184, 166, 0.55);
}

/* ---- Demo-data banner (remove once the real backend is plugged in) ---- */
.cm-demo-banner {
  text-align: center;
  font-size: 11.5px;
  font-weight: 600;
  letter-spacing: 1px;
  color: var(--cm-warn);
  background: rgba(245, 158, 11, 0.1);
  border: 1px solid rgba(245, 158, 11, 0.35);
  border-radius: 10px;
  padding: 8px 14px;
  margin: 0 0 4px 0;
}


Writing css/style.css


## 8. Write `app.py` (unchanged UI — only the backend import line changed)

Diff vs. your original: `from backend.dummy_pipeline import process` →
`from backend.pipeline import process`, and the demo-data banner now says
this is a real AI-generated report instead of placeholder data.

In [8]:
%%writefile app.py
"""
CareMate - Frontend Entry Point with Patient Report PDF Download
================================================================

Wired to the real CareMate backend (backend/pipeline.py): Whisper ASR +
the trained disease classifier. Includes a downloadable PDF for the
patient report and a "New Consultation" reset flow.
"""

import html
import os
import re
import textwrap
import time
import uuid
import zlib
from html.parser import HTMLParser

import gradio as gr

from backend.pipeline import process


CSS_PATH = os.path.join("css", "style.css")
REPORT_OUTPUT_DIR = "outputs"

EMPTY_REPORT_HTML = """
<section class="cm-report cm-report-empty">
  <h3>No report yet</h3>
  <p>Upload or record a consultation, then process it to view confidence, precautions, recommendations, and doctor-review guidance.</p>
</section>
"""


def load_css() -> str:
    if os.path.exists(CSS_PATH):
        with open(CSS_PATH, "r", encoding="utf-8") as f:
            return f.read()
    return ""


HEAD_HTML = """
<meta name="viewport" content="width=device-width, initial-scale=1">
"""


class ReportTextExtractor(HTMLParser):
    def __init__(self):
        super().__init__()
        self.parts = []

    def handle_starttag(self, tag, attrs):
        if tag in {"h1", "h2", "h3", "h4", "p", "div", "section", "article"}:
            self.parts.append("\n")
        elif tag == "li":
            self.parts.append("\n- ")

    def handle_endtag(self, tag):
        if tag in {"h1", "h2", "h3", "h4", "p", "li", "div", "section", "article"}:
            self.parts.append("\n")

    def handle_data(self, data):
        text = data.strip()
        if text:
            self.parts.append(text + " ")

    def text(self) -> str:
        raw = "".join(self.parts)
        raw = html.unescape(raw)
        raw = re.sub(r"[ \t]+", " ", raw)
        raw = re.sub(r"\n{3,}", "\n\n", raw)
        return raw.strip()


def html_to_text(report_html: str) -> str:
    parser = ReportTextExtractor()
    parser.feed(report_html or "")
    return parser.text() or "Patient report unavailable. Please consult a doctor."


def _pdf_escape(text: str) -> str:
    return text.replace("\\", "\\\\").replace("(", "\\(").replace(")", "\\)")


def _pdf_stream(lines: list[str]) -> bytes:
    commands = ["BT", "/F1 11 Tf", "72 760 Td", "15 TL"]
    first = True
    for line in lines:
        safe = _pdf_escape(line)
        if first:
            commands.append(f"({safe}) Tj")
            first = False
        else:
            commands.append(f"T* ({safe}) Tj")
    commands.append("ET")
    return "\n".join(commands).encode("latin-1", errors="replace")


def write_simple_pdf(path: str, title: str, body: str) -> str:
    """
    Writes a lightweight PDF using only the Python standard library.
    This keeps the Colab notebook free from extra PDF dependencies.
    """
    os.makedirs(os.path.dirname(path), exist_ok=True)

    lines = [title, ""]
    for paragraph in body.splitlines():
        paragraph = paragraph.strip()
        if not paragraph:
            lines.append("")
            continue
        width = 88 if not paragraph.startswith("- ") else 84
        wrapped = textwrap.wrap(paragraph, width=width)
        if paragraph.startswith("- "):
            lines.extend(wrapped)
        else:
            lines.extend(wrapped)

    if len(lines) > 43:
        lines = lines[:41] + ["", "Report continues in the CareMate UI."]

    stream = zlib.compress(_pdf_stream(lines))

    objects = [
        b"<< /Type /Catalog /Pages 2 0 R >>",
        b"<< /Type /Pages /Kids [3 0 R] /Count 1 >>",
        b"<< /Type /Page /Parent 2 0 R /MediaBox [0 0 612 792] "
        b"/Resources << /Font << /F1 4 0 R >> >> /Contents 5 0 R >>",
        b"<< /Type /Font /Subtype /Type1 /BaseFont /Helvetica >>",
        b"<< /Length " + str(len(stream)).encode("ascii") + b" /Filter /FlateDecode >>\nstream\n" + stream + b"\nendstream",
    ]

    pdf = bytearray(b"%PDF-1.4\n")
    offsets = [0]
    for idx, obj in enumerate(objects, start=1):
        offsets.append(len(pdf))
        pdf.extend(f"{idx} 0 obj\n".encode("ascii"))
        pdf.extend(obj)
        pdf.extend(b"\nendobj\n")

    xref_position = len(pdf)
    pdf.extend(f"xref\n0 {len(objects) + 1}\n".encode("ascii"))
    pdf.extend(b"0000000000 65535 f \n")
    for offset in offsets[1:]:
        pdf.extend(f"{offset:010d} 00000 n \n".encode("ascii"))
    pdf.extend(
        f"trailer\n<< /Size {len(objects) + 1} /Root 1 0 R >>\nstartxref\n{xref_position}\n%%EOF\n".encode(
            "ascii"
        )
    )

    with open(path, "wb") as f:
        f.write(pdf)
    return path


def create_report_pdf(
    detected_language: str,
    transcript: str,
    standardized_transcript: str,
    translated_transcript: str,
    clinical_summary: str,
    clinical_report_html: str,
) -> str:
    report_text = html_to_text(clinical_report_html)
    pdf_body = f"""Detected language: {detected_language}

Clinical summary:
{clinical_summary}

Patient report:
{report_text}

Original transcript:
{transcript}

Standardized transcript:
{standardized_transcript}

Translated transcript:
{translated_transcript}
"""
    # IMPORTANT: use a unique filename per run. Gradio/browsers cache
    # file downloads by path, so reusing the same filename on every
    # consultation can make the "Download PDF" button keep serving the
    # *previous* patient's report instead of refreshing with the new one.
    unique_name = f"caremate_patient_report_{int(time.time())}_{uuid.uuid4().hex[:6]}.pdf"
    pdf_path = os.path.join(REPORT_OUTPUT_DIR, unique_name)
    return write_simple_pdf(pdf_path, "CareMate Patient Report", pdf_body)


def fallback_report_html(clinical_summary: str) -> str:
    safe_summary = html.escape(clinical_summary or "Doctor review required.")
    return f"""
    <section class="cm-report">
      <div class="cm-report-header">
        <div>
          <span class="cm-report-kicker">Patient Report</span>
          <h3>Clinical Review Suggested</h3>
        </div>
        <div class="cm-score">
          <span>Confidence</span>
          <strong>--</strong>
        </div>
      </div>
      <div class="cm-confidence" aria-label="Disease confidence score unavailable">
        <span style="width: 35%"></span>
      </div>
      <div class="cm-report-grid">
        <article>
          <span class="cm-chip cm-chip-blue">Summary</span>
          <p>{safe_summary}</p>
        </article>
        <article>
          <span class="cm-chip cm-chip-amber">Priority</span>
          <p>Please consult a doctor for proper diagnosis and treatment.</p>
        </article>
      </div>
      <div class="cm-report-columns">
        <div>
          <h4>Precautions</h4>
          <ul>
            <li>Rest, hydrate, and monitor symptoms.</li>
            <li>Seek urgent care if symptoms become severe or unusual.</li>
          </ul>
        </div>
        <div>
          <h4>Recommendations</h4>
          <ul>
            <li>Share this summary with a qualified clinician.</li>
            <li>Do not self-medicate based only on AI output.</li>
          </ul>
        </div>
      </div>
      <div class="cm-disclaimer">
        This AI report is not a diagnosis. Please visit a qualified doctor for confirmation and treatment.
      </div>
    </section>
    """


def run_pipeline(audio_path, output_language):
    if not audio_path:
        raise gr.Error("Please upload an audio consultation first.")

    result = process(audio_path, output_language)

    if len(result) == 5:
        (
            detected_language,
            transcript,
            standardized_transcript,
            translated_transcript,
            clinical_summary,
        ) = result
        clinical_report_html = fallback_report_html(clinical_summary)
    else:
        (
            detected_language,
            transcript,
            standardized_transcript,
            translated_transcript,
            clinical_summary,
            clinical_report_html,
        ) = result

    report_pdf_path = create_report_pdf(
        detected_language,
        transcript,
        standardized_transcript,
        translated_transcript,
        clinical_summary,
        clinical_report_html,
    )

    return (
        detected_language,
        transcript,
        standardized_transcript,
        translated_transcript,
        clinical_summary,
        clinical_report_html,
        # DownloadButton: set the file AND make the button visible/clickable
        # in one shot, so it's a single real click to download, no file-card
        # detour and no separate "reveal" step.
        gr.DownloadButton(value=report_pdf_path, visible=True),
    )


def reset_consultation():
    """Clears every field so the next audio input starts from a blank slate."""
    return (
        None,          # audio_input
        "",            # detected_language
        "",            # transcript
        "",            # standardized_transcript
        "",            # translated_transcript
        "",            # clinical_summary
        EMPTY_REPORT_HTML,  # clinical_report
        gr.DownloadButton(value=None, visible=False),  # report_pdf
    )


theme = gr.themes.Soft(
    primary_hue="blue",
    secondary_hue="sky",
    neutral_hue="slate",
).set(
    body_background_fill="*neutral_950",
    block_background_fill="rgba(15,23,42,0.0)",
    block_border_width="0px",
    block_shadow="none",
)


with gr.Blocks(theme=theme, css=load_css(), head=HEAD_HTML, title="CareMate") as demo:
    with gr.Column(elem_id="cm-root"):
        gr.HTML(
            """
            <div id="cm-header">
                <div id="cm-title">CAREMATE</div>
                <div id="cm-subtitle">AI Powered Multilingual Clinical Assistant</div>
            </div>
            """
        )

        with gr.Row(elem_id="cm-input-row"):
            with gr.Column(scale=2, elem_classes=["cm-card"]):
                gr.HTML('<span class="cm-section-label">Upload Consultation Audio</span>')
                audio_input = gr.Audio(
                    sources=["upload"],
                    type="filepath",
                    label="",
                    show_label=False,
                    elem_id="cm-audio",
                )

            with gr.Column(scale=1, elem_classes=["cm-card"]):
                gr.HTML('<span class="cm-section-label">Output Language</span>')
                output_language = gr.Radio(
                    choices=["ENGLISH", "HINDI"],
                    value="ENGLISH",
                    label="",
                    elem_id="cm-lang-selector",
                    container=False,
                )

            with gr.Column(scale=1, elem_classes=["cm-card"]):
                gr.HTML('<span class="cm-section-label">Detected Language</span>')
                detected_language = gr.Textbox(
                    show_label=False,
                    interactive=False,
                    elem_id="cm-detected-lang",
                    elem_classes=["cm-output"],
                    lines=1,
                )

        with gr.Row(elem_id="cm-process-row"):
            process_btn = gr.Button("Process Consultation", elem_id="cm-process-btn", variant="primary")
            reset_btn = gr.Button("🔄 New Consultation", elem_id="cm-reset-btn", variant="secondary")

        with gr.Row(elem_id="cm-outputs-row"):
            with gr.Column(elem_classes=["cm-card"]):
                gr.HTML('<span class="cm-section-label">Original Transcript</span>')
                transcript = gr.Textbox(
                    show_label=False,
                    interactive=False,
                    elem_classes=["cm-output"],
                    lines=6,
                )

            with gr.Column(elem_classes=["cm-card"]):
                gr.HTML(
                    '<span class="cm-section-label">Standardized Clinical Transcript</span>'
                )
                standardized_transcript = gr.Textbox(
                    show_label=False,
                    interactive=False,
                    elem_classes=["cm-output"],
                    lines=6,
                )

            with gr.Column(elem_classes=["cm-card"]):
                gr.HTML('<span class="cm-section-label">Translated Transcript</span>')
                translated_transcript = gr.Textbox(
                    show_label=False,
                    interactive=False,
                    elem_classes=["cm-output"],
                    lines=6,
                )

            with gr.Column(elem_classes=["cm-card"]):
                gr.HTML('<span class="cm-section-label">Clinical Summary</span>')
                clinical_summary = gr.Textbox(
                    show_label=False,
                    interactive=False,
                    elem_classes=["cm-output"],
                    lines=6,
                )

        with gr.Column(elem_classes=["cm-card"], elem_id="cm-report-card"):
            gr.HTML('<span class="cm-section-label">Clinical Report</span>')
            gr.HTML(
                '<div class="cm-demo-banner">This report is AI-generated from your audio. '
                'It is not a diagnosis — always confirm with a qualified doctor.</div>'
            )
            clinical_report = gr.HTML(EMPTY_REPORT_HTML)
            report_pdf = gr.DownloadButton(
                label="Download Patient Report PDF",
                elem_id="cm-download-btn",
                visible=False,
            )

        gr.HTML(
            '<div id="cm-footer">CareMate &middot; Clinical Decision Support &middot; Doctor Review Required</div>'
        )

    all_outputs = [
        detected_language,
        transcript,
        standardized_transcript,
        translated_transcript,
        clinical_summary,
        clinical_report,
        report_pdf,
    ]

    process_btn.click(
        fn=run_pipeline,
        inputs=[audio_input, output_language],
        outputs=all_outputs,
    )

    reset_btn.click(
        fn=reset_consultation,
        inputs=None,
        outputs=[audio_input] + all_outputs,
    )


if __name__ == "__main__":
    demo.launch()


Writing app.py


## 9. Write `requirements.txt` and `README.md`

In [9]:
%%writefile requirements.txt
gradio==5.38.2
openai-whisper
transformers
sentencepiece
accelerate
joblib
pandas
torch

Writing requirements.txt


In [10]:
%%writefile README.md
---
title: CareMate
emoji: 🩺
colorFrom: blue
colorTo: gray
sdk: gradio
sdk_version: 5.38.2
app_file: app.py
pinned: false
---

# CareMate

AI-powered multilingual clinical assistant. Upload a patient consultation
audio clip; the app transcribes it (Whisper), predicts the most likely
conditions from the symptoms (your trained classifier), and generates a
patient report with a downloadable PDF.

## Project structure

```
frontend/
+-- app.py                  # Gradio UI - entry point
+-- requirements.txt
+-- README.md
+-- css/
|   +-- style.css           # Dark glass "clinical dashboard" theme
+-- backend/
|   +-- __init__.py
|   +-- pipeline.py         # Real backend: Whisper + disease classifier
+-- assets/
+-- samples/
+-- outputs/                # Saved patient reports (JSON) + history CSV
+-- components/
```

The classifier itself lives one level up, at `../models/Best_Disease_Model`
(i.e. `CareMate_AI/models/Best_Disease_Model`), matching where the
deployment notebook trained and saved it.

## Backend contract

```python
from backend.pipeline import process

detected_language, transcript, standardized_transcript, translated_transcript, clinical_summary, clinical_report_html = process(
    audio_path, output_language
)
```

## Known limitations (see backend/pipeline.py docstring for detail)

- `standardized_transcript` is the English text fed to the classifier, not
  output of a separately trained clinical-NLP normalizer (none exists yet).
- Hindi output translation is only genuine when the source audio is
  already Hindi or English is requested; otherwise it falls back to
  English text with a clear note, since no general translation model is
  wired in yet.

## How to run

Just run this notebook top to bottom in Colab with Drive mounted and your
trained model already saved at `CareMate_AI/models/Best_Disease_Model`.
The last cell launches a public Gradio link via `share=True`.


Writing README.md


## 10. Launch the app on real input

In [12]:
# Fresh import so edits above are picked up if you re-run cells.
import importlib
import sys

for mod_name in ["app", "backend.pipeline"]:
    if mod_name in sys.modules:
        del sys.modules[mod_name]

import app as caremate_app
caremate_app.demo.launch(share=True, debug=True)

CareMate backend: loading models...
Loading Whisper (base)...
Whisper loaded.
Loading disease classifier from: /content/drive/MyDrive/CareMate_AI/models/Best_Disease_Model


OSError: Repo id must be in the form 'repo_name' or 'namespace/repo_name': '/content/drive/MyDrive/CareMate_AI/models/Best_Disease_Model'. Use `repo_type` argument if needed.

## Notes on re-running

- If you edit `backend/pipeline.py` or `app.py` after first launch, re-run
  the corresponding `%%writefile` cell, then re-run the launch cell — it
  clears the module cache first so your changes actually take effect.
- Model loading (Whisper + classifier) happens once, when `backend/pipeline.py`
  is imported — expect a short delay the first time you launch.
- Saved reports land in `frontend/outputs/*.json`, and a running
  `frontend/outputs/patient_history.csv` log is kept alongside them.